In [2]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import open3d as o3d


# ============================================================
# CONFIGURACIÓN
# ============================================================

CARPETA_CSV = "./pointclouds"

ARCHIVO_VIDEO = "./tracking_video_3D.mp4"

FPS = 10

# Detección
VOXEL_SIZE = 0.08
DBSCAN_EPS = 0.6
DBSCAN_MIN_POINTS = 10

RANSAC_DISTANCE = 0.15
RANSAC_ITERATIONS = 80

# Tracking
MAX_TRACK_DISTANCE = 2.0
MAX_MISSING = 3


# ============================================================
# CÁMARA
# ============================================================

# Vista diagonal:
# - componente X/Y -> lateral/frontal
# - componente Z -> altura de cámara

CAMERA_FRONT = [-0.60, -0.40, -0.25]

CAMERA_UP = [0.0, 0.0, 1.0]

CAMERA_ZOOM = 0.35


# ============================================================
# COLORES
# ============================================================

COLORES = [
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.5, 1.0],
    [1.0, 1.0, 0.0],
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.5, 0.0],
    [0.5, 0.0, 1.0],
    [0.5, 1.0, 0.0],
    [1.0, 0.0, 0.5],
]


# ============================================================
# DETECCIÓN
# ============================================================

def detectar_frame(archivo):

    df = pd.read_csv(archivo)

    original_points = df[
        ["x", "y", "z"]
    ].to_numpy(
        dtype=np.float64
    )

    # Eliminar puntos cercanos al origen
    original_points = original_points[
        np.linalg.norm(
            original_points,
            axis=1
        ) > 0.1
    ]

    points = original_points.copy()

    if len(points) < 10:

        return (
            original_points,
            np.empty((0, 3)),
            np.array([]),
            []
        )


    # --------------------------------------------------------
    # FILTRO ESPACIAL
    # --------------------------------------------------------

    x = points[:, 0]
    y = points[:, 1]
    z = points[:, 2]

    xmin, xmax = np.percentile(
        x,
        [0, 100]
    )

    ymin, ymax = np.percentile(
        y,
        [3, 100]
    )

    zmin, zmax = np.percentile(
        z,
        [20, 85]
    )

    mask = (
        (x >= xmin) &
        (x <= xmax) &
        (y >= ymin) &
        (y <= ymax) &
        (z >= zmin) &
        (z <= zmax)
    )

    points = points[mask]


    # --------------------------------------------------------
    # OPEN3D
    # --------------------------------------------------------

    pcd = o3d.geometry.PointCloud()

    pcd.points = (
        o3d.utility.Vector3dVector(
            points
        )
    )


    # --------------------------------------------------------
    # OUTLIERS
    # --------------------------------------------------------

    if len(pcd.points) > 20:

        pcd, _ = pcd.remove_statistical_outlier(
            nb_neighbors=20,
            std_ratio=2.0
        )


    # --------------------------------------------------------
    # VOXEL
    # --------------------------------------------------------

    if len(pcd.points) > 0:

        pcd_down = pcd.voxel_down_sample(
            VOXEL_SIZE
        )

    else:

        pcd_down = pcd


    if len(pcd_down.points) < 3:

        return (
            original_points,
            np.empty((0, 3)),
            np.array([]),
            []
        )


    # --------------------------------------------------------
    # RANSAC
    # --------------------------------------------------------

    try:

        plane_model, ground_indices = (
            pcd_down.segment_plane(
                distance_threshold=RANSAC_DISTANCE,
                ransac_n=3,
                num_iterations=RANSAC_ITERATIONS
            )
        )

    except:

        ground_indices = []


    objects = pcd_down.select_by_index(
        ground_indices,
        invert=True
    )


    # --------------------------------------------------------
    # DBSCAN
    # --------------------------------------------------------

    if len(objects.points) > 0:

        labels = np.array(
            objects.cluster_dbscan(
                eps=DBSCAN_EPS,
                min_points=DBSCAN_MIN_POINTS,
                print_progress=False
            )
        )

    else:

        labels = np.array([])


    # --------------------------------------------------------
    # BOUNDING BOXES
    # --------------------------------------------------------

    boxes = []

    if len(labels) > 0:

        max_label = labels.max()

        for cluster_id in range(
            max_label + 1
        ):

            indices = np.where(
                labels == cluster_id
            )[0]

            if len(indices) == 0:
                continue

            cluster = objects.select_by_index(
                indices
            )

            box = cluster.get_axis_aligned_bounding_box()

            boxes.append(
                (
                    cluster_id,
                    box
                )
            )


    return (
        original_points,
        np.asarray(objects.points),
        labels,
        boxes
    )


# ============================================================
# TRACKER
# ============================================================

tracks = {}

next_track_id = 0


def actualizar_tracking(centers):

    global tracks
    global next_track_id

    asignaciones = {}

    ids_previos = list(
        tracks.keys()
    )

    usados = set()


    # --------------------------------------------------------
    # ASIGNAR VEHÍCULOS
    # --------------------------------------------------------

    for i, centro in enumerate(
        centers
    ):

        mejor_id = None
        mejor_distancia = float("inf")


        for track_id in ids_previos:

            if track_id in usados:
                continue

            centro_anterior = tracks[
                track_id
            ]["center"]

            distancia = np.linalg.norm(
                centro -
                centro_anterior
            )


            if (
                distancia <
                mejor_distancia
                and
                distancia <
                MAX_TRACK_DISTANCE
            ):

                mejor_distancia = distancia
                mejor_id = track_id


        # Vehículo conocido
        if mejor_id is not None:

            asignaciones[i] = mejor_id

            usados.add(
                mejor_id
            )

        # Vehículo nuevo
        else:

            nuevo_id = next_track_id

            next_track_id += 1

            tracks[nuevo_id] = {
                "center": centro.copy(),
                "missing": 0
            }

            asignaciones[i] = nuevo_id

            usados.add(
                nuevo_id
            )


    # --------------------------------------------------------
    # ACTUALIZAR TRACKS
    # --------------------------------------------------------

    nuevos_tracks = {}


    for i, track_id in asignaciones.items():

        nuevos_tracks[track_id] = {
            "center": centers[i].copy(),
            "missing": 0
        }


    # Vehículos temporalmente desaparecidos
    for track_id in ids_previos:

        if track_id not in usados:

            missing = (
                tracks[track_id]["missing"]
                + 1
            )

            if missing <= MAX_MISSING:

                nuevos_tracks[track_id] = {
                    "center": tracks[track_id]["center"],
                    "missing": missing
                }


    tracks = nuevos_tracks

    return asignaciones


# ============================================================
# BUSCAR CSV
# ============================================================

archivos = sorted(
    glob.glob(
        os.path.join(
            CARPETA_CSV,
            "*.csv"
        )
    )
)


print()
print(
    "CSV encontrados:",
    len(archivos)
)
print()


if len(archivos) == 0:

    raise RuntimeError(
        "No hay CSV dentro de ./pointclouds"
    )


# ============================================================
# PROCESAR TODOS LOS FRAMES
# ============================================================

frames = []

todos_puntos = []


for numero, archivo in enumerate(
    archivos
):

    print(
        f"Procesando "
        f"{numero + 1}/{len(archivos)}: "
        f"{os.path.basename(archivo)}"
    )


    (
        original_points,
        object_points,
        labels,
        boxes
    ) = detectar_frame(
        archivo
    )


    # Guardamos puntos para determinar
    # el centro global de la escena
    if len(original_points) > 0:

        todos_puntos.append(
            original_points
        )


    # --------------------------------------------------------
    # CENTROS
    # --------------------------------------------------------

    centers = []

    for cluster_id, box in boxes:

        centers.append(
            np.asarray(
                box.get_center()
            )
        )


    if len(centers) > 0:

        centers = np.asarray(
            centers
        )

    else:

        centers = np.empty(
            (0, 3)
        )


    # --------------------------------------------------------
    # TRACKING
    # --------------------------------------------------------

    asignaciones = actualizar_tracking(
        centers
    )


    # --------------------------------------------------------
    # BOXES + ID
    # --------------------------------------------------------

    boxes_track = []

    for i, (
        cluster_id,
        box
    ) in enumerate(boxes):

        track_id = asignaciones[i]

        boxes_track.append(
            (
                track_id,
                box
            )
        )


    frames.append(
        {
            "points": original_points,
            "boxes": boxes_track
        }
    )


# ============================================================
# CENTRO GLOBAL
# ============================================================

todos_puntos = np.vstack(
    todos_puntos
)


scene_min = todos_puntos.min(
    axis=0
)

scene_max = todos_puntos.max(
    axis=0
)

scene_center = (
    scene_min +
    scene_max
) / 2


print()
print(
    "Centro de la escena:",
    scene_center
)


# ============================================================
# VISUALIZADOR
# ============================================================

vis = o3d.visualization.Visualizer()


vis.create_window(
    window_name="Tracking 3D",
    width=1280,
    height=720,
    visible=True
)


# ============================================================
# NUBE
# ============================================================

cloud = o3d.geometry.PointCloud()

vis.add_geometry(
    cloud
)


# ============================================================
# CREAR UNA CAJA DE REFERENCIA
# ============================================================

# Esto ayuda a que Open3D conozca desde el principio
# el tamaño de la escena.

bbox_scene = o3d.geometry.AxisAlignedBoundingBox(
    scene_min,
    scene_max
)

bbox_scene.color = [
    0.0,
    0.0,
    0.0
]

vis.add_geometry(
    bbox_scene
)


# ============================================================
# CONFIGURAR CÁMARA
# ============================================================

vis.poll_events()

vis.update_renderer()


view = vis.get_view_control()


view.set_lookat(
    scene_center
)

view.set_front(
    CAMERA_FRONT
)

view.set_up(
    CAMERA_UP
)

view.set_zoom(
    CAMERA_ZOOM
)


# ============================================================
# VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)


video = cv2.VideoWriter(
    ARCHIVO_VIDEO,
    fourcc,
    FPS,
    (1280, 720)
)


# ============================================================
# GENERAR FRAMES
# ============================================================

print()
print(
    "Generando vídeo..."
)
print()


for numero, frame in enumerate(
    frames
):

    print(
        f"Frame "
        f"{numero + 1}/{len(frames)}"
    )


    # --------------------------------------------------------
    # ACTUALIZAR NUBE
    # --------------------------------------------------------

    puntos = frame[
        "points"
    ]


    if len(puntos) > 0:

        cloud.points = (
            o3d.utility.Vector3dVector(
                puntos
            )
        )


        # Gris
        colores = np.full(
            (
                len(puntos),
                3
            ),
            [0.55, 0.55, 0.55]
        )


        cloud.colors = (
            o3d.utility.Vector3dVector(
                colores
            )
        )


    vis.update_geometry(
        cloud
    )


    # --------------------------------------------------------
    # CAJAS
    # --------------------------------------------------------

    cajas_creadas = []


    for track_id, box in frame[
        "boxes"
    ]:

        box.color = COLORES[
            track_id %
            len(COLORES)
        ]


        vis.add_geometry(
            box,
            reset_bounding_box=False
        )


        cajas_creadas.append(
            box
        )


    # --------------------------------------------------------
    # IMPORTANTE:
    # NO DEJAR QUE OPEN3D CAMBIE LA CÁMARA
    # --------------------------------------------------------

    view.set_lookat(
        scene_center
    )

    view.set_front(
        CAMERA_FRONT
    )

    view.set_up(
        CAMERA_UP
    )

    view.set_zoom(
        CAMERA_ZOOM
    )


    # --------------------------------------------------------
    # RENDER
    # --------------------------------------------------------

    vis.poll_events()

    vis.update_renderer()


    imagen = (
        vis.capture_screen_float_buffer(
            do_render=True
        )
    )


    imagen = np.asarray(
        imagen
    )


    imagen = (
        imagen * 255
    ).astype(
        np.uint8
    )


    imagen = cv2.cvtColor(
        imagen,
        cv2.COLOR_RGB2BGR
    )


    video.write(
        imagen
    )


    # --------------------------------------------------------
    # QUITAR CAJAS
    # --------------------------------------------------------

    for box in cajas_creadas:

        vis.remove_geometry(
            box,
            reset_bounding_box=False
        )


# ============================================================
# CERRAR
# ============================================================

video.release()

vis.destroy_window()


print()
print(
    "======================================"
)

print(
    "VÍDEO TERMINADO"
)

print(
    "Archivo:",
    ARCHIVO_VIDEO
)

print(
    "======================================"
)


CSV encontrados: 369

Procesando 1/369: pointcloud_1727346186_488981170.csv
Procesando 2/369: pointcloud_1727346186_538003836.csv
Procesando 3/369: pointcloud_1727346186_586942416.csv
Procesando 4/369: pointcloud_1727346186_638212608.csv
Procesando 5/369: pointcloud_1727346186_688448845.csv
Procesando 6/369: pointcloud_1727346186_738318465.csv
Procesando 7/369: pointcloud_1727346186_789278730.csv
Procesando 8/369: pointcloud_1727346186_838301348.csv
Procesando 9/369: pointcloud_1727346186_887269654.csv
Procesando 10/369: pointcloud_1727346186_936946076.csv
Procesando 11/369: pointcloud_1727346186_987057480.csv
Procesando 12/369: pointcloud_1727346187_138756258.csv
Procesando 13/369: pointcloud_1727346187_188181141.csv
Procesando 14/369: pointcloud_1727346187_237884728.csv
Procesando 15/369: pointcloud_1727346187_288974464.csv
Procesando 16/369: pointcloud_1727346187_336883338.csv
Procesando 17/369: pointcloud_1727346187_386856659.csv
Procesando 18/369: pointcloud_1727346187_39143223.c